# Inspect the generalized U-Net heatmap prediction

This notebook loads the generalized model's `best_UNet.pt` when available, otherwise `latest_UNet.pt`, and displays:

1. The RGB camera observation.
2. The model's predicted heatmap probability.
3. The environment's ground-truth heatmap.
4. The prediction overlaid on the RGB observation.

It runs in evaluation mode with gradients disabled and does not modify the saved training seed or iteration.

In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn.functional as F


def find_environment_directory():
    current_directory = Path.cwd().resolve()
    candidates = [
        current_directory,
        current_directory / "CODE" / "Enviornment",
        *current_directory.parents,
    ]
    for candidate in candidates:
        if (
            (candidate / "enviornment.py").is_file()
            and (candidate / "Model").is_dir()
            and (candidate / "xml_models" / "world.xml").is_file()
        ):
            return candidate
    raise RuntimeError("Could not find CODE/Enviornment from the current notebook directory")


ENVIRONMENT_DIRECTORY = find_environment_directory()
if str(ENVIRONMENT_DIRECTORY) not in sys.path:
    sys.path.insert(0, str(ENVIRONMENT_DIRECTORY))

from Model.UNet.unet import UNet
from Model.UNet.tmp.UNET_MODEL_OVERFIT_2ND_ROUND.UNetV2Constants import UNetV2Constants
from enviornment import Enviornment
from enviornment_randomizer import Enviornment_Randomizer
from randomization_constants import Randomization_Constants

WORLD_XML_FILE = ENVIRONMENT_DIRECTORY / "xml_models" / "world.xml"
CHECKPOINT_DIRECTORY = ENVIRONMENT_DIRECTORY / "Model" / "UNet" / "tmp" / "General_Attempt_Using_2nd_Overfit"
BEST_CHECKPOINT_FILE = CHECKPOINT_DIRECTORY / "best_UNet.pt"
LATEST_CHECKPOINT_FILE = CHECKPOINT_DIRECTORY / "latest_UNet.pt"
CHECKPOINT_FILE = BEST_CHECKPOINT_FILE if BEST_CHECKPOINT_FILE.is_file() else LATEST_CHECKPOINT_FILE
LAST_TRAINED_SEED_FILE = CHECKPOINT_DIRECTORY / "last_seed.txt"

print(f"Environment directory: {ENVIRONMENT_DIRECTORY}")
print(f"Checkpoint: {CHECKPOINT_FILE}")

In [ ]:
if not CHECKPOINT_FILE.is_file():
    raise RuntimeError(
        f"Checkpoint not found: {CHECKPOINT_FILE}\n"
        "Run the generalized trainer until latest_UNet.pt or best_UNet.pt is created."
    )

model_constants = UNetV2Constants()
randomization_constants = Randomization_Constants()
enviornment_randomizer = Enviornment_Randomizer()

env = Enviornment(
    xml_file=str(WORLD_XML_FILE),
    Enviornment_Randomizer=enviornment_randomizer,
    Randomization_Constants=randomization_constants,
    Model_Constants=model_constants,
)

model = UNet(
    in_channels=3,
    num_classes=1,
    checkpoint_dir=str(CHECKPOINT_DIRECTORY),
    learning_rate=model_constants.learning_rate,
)
model.load_state_dict(
    torch.load(CHECKPOINT_FILE, map_location=model.device, weights_only=True)
)
model.eval()

print(f"Loaded checkpoint on: {model.device}")

In [ ]:
def prepare_scene(seed):
    env.new_scene(seed=seed)
    rgb_observation = env.observation()
    target_heatmap = env.get_target_heatmap()

    model_input = torch.from_numpy(rgb_observation).permute(2, 0, 1).unsqueeze(0).float()
    model_input = model_input / 255.0
    model_input = F.interpolate(
        model_input,
        size=(model_constants.input_y_dim, model_constants.input_x_dim),
        mode="bilinear",
        align_corners=False,
    )
    model_input = model_input.to(model.device)
    return rgb_observation, model_input, target_heatmap


def predict_heatmap(selected_model, model_input):
    selected_model.eval()

    with torch.no_grad():
        prediction_logits = selected_model(model_input)
        predicted_heatmap = torch.sigmoid(prediction_logits)[0, 0].cpu().numpy()
    return predicted_heatmap


def predict_scene(seed, selected_model=model):
    rgb_observation, model_input, target_heatmap = prepare_scene(seed)
    predicted_heatmap = predict_heatmap(selected_model, model_input)

    return rgb_observation, predicted_heatmap, target_heatmap


In [ ]:
# Test well beyond the last seed used for generalized training.
last_trained_seed = int(LAST_TRAINED_SEED_FILE.read_text().strip())
TEST_SEED = last_trained_seed + 1_000_000 +np.random.randint(1, 1012)
rgb_observation, predicted_heatmap, target_heatmap = predict_scene(TEST_SEED)

predicted_peak_y, predicted_peak_x = np.unravel_index(
    np.argmax(predicted_heatmap),
    predicted_heatmap.shape,
)
target_peak_y, target_peak_x = np.unravel_index(
    np.argmax(target_heatmap),
    target_heatmap.shape,
)

print(f"Test seed: {TEST_SEED}")
print(f"RGB shape: {rgb_observation.shape}")
print(f"Prediction shape: {predicted_heatmap.shape}")
print(f"Target shape: {target_heatmap.shape}")
print(f"Prediction range: {predicted_heatmap.min():.6f} to {predicted_heatmap.max():.6f}")
print(f"Target range: {target_heatmap.min():.6f} to {target_heatmap.max():.6f}")
print(f"Strongest predicted pixel (x, y): ({predicted_peak_x}, {predicted_peak_y})")
print(f"Strongest target pixel (x, y): ({target_peak_x}, {target_peak_y})")

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(22, 5), constrained_layout=True)

observation_height, observation_width = rgb_observation.shape[:2]
heatmap_height, heatmap_width = predicted_heatmap.shape
predicted_peak_rgb_x = (predicted_peak_x + 0.5) * observation_width / heatmap_width
predicted_peak_rgb_y = (predicted_peak_y + 0.5) * observation_height / heatmap_height

axes[0].imshow(rgb_observation)
axes[0].scatter(
    predicted_peak_rgb_x,
    predicted_peak_rgb_y,
    c="cyan",
    marker="x",
    s=120,
    linewidths=2,
)
axes[0].set_title(f"RGB observation — seed {TEST_SEED}")
axes[0].axis("off")

prediction_image = axes[1].imshow(
    predicted_heatmap,
    cmap="magma",
    vmin=0.0,
    vmax=1.0,
    origin="upper",
)
axes[1].scatter(predicted_peak_x, predicted_peak_y, c="cyan", marker="x", s=80)
axes[1].set_title("Model prediction")
axes[1].set_xlabel("Heatmap x")
axes[1].set_ylabel("Heatmap y")
fig.colorbar(prediction_image, ax=axes[1], label="Predicted probability")

target_image = axes[2].imshow(
    target_heatmap,
    cmap="magma",
    vmin=0.0,
    vmax=1.0,
    origin="upper",
)
axes[2].scatter(target_peak_x, target_peak_y, c="cyan", marker="x", s=80)
axes[2].set_title("Ground-truth heatmap")
axes[2].set_xlabel("Heatmap x")
axes[2].set_ylabel("Heatmap y")
fig.colorbar(target_image, ax=axes[2], label="Target value")

axes[3].imshow(rgb_observation)
axes[3].imshow(
    predicted_heatmap,
    cmap="magma",
    alpha=0.55,
    vmin=0.0,
    vmax=1.0,
    origin="upper",
    extent=(0, observation_width, observation_height, 0),
)
axes[3].set_title("Prediction overlay")
axes[3].axis("off")

plt.show()

## Try another scene

Change `TEST_SEED` to inspect another deterministic scene and rerun the final two cells. Keep it above `last_trained_seed` when you want an unseen scene. The notebook does not modify the generalized trainer's seed or iteration files.

## Compare every saved model on 1,000 unseen random scenes

This benchmark discovers every saved `.pt` checkpoint under `Model/UNet/tmp`, loads all compatible models, and evaluates them on the exact same 1,000 distinct random seeds well beyond the training range. Each scene is rendered once and passed to every model. The main localization metric is the distance from the strongest predicted peak to the nearest target center; this does not by itself prove that every object in a multi-object scene was detected.

In [ ]:
VALIDATION_COUNT = 1_000
VALIDATION_RANDOM_SEED = 20_260_726
VALIDATION_START_SEED = last_trained_seed + 2_000_000
VALIDATION_SEED_POOL_SIZE = 10_000_000

checkpoint_root = ENVIRONMENT_DIRECTORY / "Model" / "UNet" / "tmp"
checkpoint_files = sorted(checkpoint_root.rglob("*.pt"))
if not checkpoint_files:
    raise RuntimeError(f"No model checkpoints found under {checkpoint_root}")

comparison_models = {}
for checkpoint_file in checkpoint_files:
    model_name = str(checkpoint_file.relative_to(checkpoint_root).with_suffix(""))
    if checkpoint_file.resolve() == CHECKPOINT_FILE.resolve():
        comparison_model = model
    else:
        comparison_model = UNet(
            in_channels=3,
            num_classes=1,
            checkpoint_dir=str(checkpoint_file.parent),
            learning_rate=model_constants.learning_rate,
        )
        comparison_model.load_state_dict(
            torch.load(
                checkpoint_file,
                map_location=comparison_model.device,
                weights_only=True,
            )
        )
        comparison_model.eval()
    comparison_models[model_name] = comparison_model

validation_rng = np.random.default_rng(VALIDATION_RANDOM_SEED)
validation_seeds = VALIDATION_START_SEED + validation_rng.choice(
    VALIDATION_SEED_POOL_SIZE,
    size=VALIDATION_COUNT,
    replace=False,
)

comparison_results = {
    model_name: {
        "peak_distances": [],
        "target_center_probabilities": [],
    }
    for model_name in comparison_models
}
evaluated_seeds = []

print(f"Comparing {len(comparison_models)} models on {VALIDATION_COUNT} scenes")
for validation_index, validation_seed in enumerate(validation_seeds, start=1):
    _, model_input, validation_target = prepare_scene(int(validation_seed))
    target_centers = np.argwhere(np.isclose(validation_target, 1.0))
    if len(target_centers) == 0:
        continue

    evaluated_seeds.append(int(validation_seed))
    for model_name, comparison_model in comparison_models.items():
        validation_prediction = predict_heatmap(comparison_model, model_input)
        prediction_peak_y, prediction_peak_x = np.unravel_index(
            np.argmax(validation_prediction),
            validation_prediction.shape,
        )
        distances_to_targets = np.sqrt(
            (target_centers[:, 0] - prediction_peak_y) ** 2
            + (target_centers[:, 1] - prediction_peak_x) ** 2
        )
        target_center_probability = validation_prediction[
            target_centers[:, 0],
            target_centers[:, 1],
        ].mean()

        comparison_results[model_name]["peak_distances"].append(
            float(distances_to_targets.min())
        )
        comparison_results[model_name]["target_center_probabilities"].append(
            float(target_center_probability)
        )

    if validation_index % 50 == 0:
        print(f"Evaluated {validation_index}/{VALIDATION_COUNT} scenes")

if not evaluated_seeds:
    raise RuntimeError("Validation produced no scenes with target centers")

for model_result in comparison_results.values():
    model_result["peak_distances"] = np.asarray(model_result["peak_distances"])
    model_result["target_center_probabilities"] = np.asarray(
        model_result["target_center_probabilities"]
    )

print(f"Random-seed generator seed: {VALIDATION_RANDOM_SEED}")
print(f"Scenes evaluated per model: {len(evaluated_seeds)}")
print()
for model_name, model_result in comparison_results.items():
    distances = model_result["peak_distances"]
    probabilities = model_result["target_center_probabilities"]
    print(model_name)
    print(f"  Mean / median distance: {distances.mean():.3f} / {np.median(distances):.3f} pixels")
    print(f"  95th percentile / worst: {np.percentile(distances, 95):.3f} / {distances.max():.3f} pixels")
    print(f"  Within one pixel: {np.mean(distances <= 1.0):.1%}")
    print(f"  Mean target-center probability: {probabilities.mean():.4f}")


In [ ]:
model_names = list(comparison_results)
short_model_names = [name.replace("General_Attempt_Using_2nd_Overfit/", "General/") for name in model_names]
mean_distances = [comparison_results[name]["peak_distances"].mean() for name in model_names]
median_distances = [np.median(comparison_results[name]["peak_distances"]) for name in model_names]
within_one_pixel_rates = [np.mean(comparison_results[name]["peak_distances"] <= 1.0) for name in model_names]
mean_target_probabilities = [comparison_results[name]["target_center_probabilities"].mean() for name in model_names]

fig, validation_axes = plt.subplots(
    2,
    2,
    figsize=(18, 12),
    constrained_layout=True,
)

x_positions = np.arange(len(model_names))
bar_width = 0.38
validation_axes[0, 0].bar(x_positions - bar_width / 2, mean_distances, bar_width, label="Mean")
validation_axes[0, 0].bar(x_positions + bar_width / 2, median_distances, bar_width, label="Median")
validation_axes[0, 0].set_title("Peak localization error — lower is better")
validation_axes[0, 0].set_ylabel("Heatmap pixels")
validation_axes[0, 0].set_xticks(x_positions, short_model_names, rotation=25, ha="right")
validation_axes[0, 0].legend()

validation_axes[0, 1].bar(short_model_names, within_one_pixel_rates, color="tab:green")
validation_axes[0, 1].set_ylim(0.0, 1.0)
validation_axes[0, 1].set_title("Predicted peak within one pixel — higher is better")
validation_axes[0, 1].set_ylabel("Fraction of scenes")
validation_axes[0, 1].tick_params(axis="x", rotation=25)

validation_axes[1, 0].bar(short_model_names, mean_target_probabilities, color="tab:orange")
validation_axes[1, 0].set_ylim(0.0, 1.0)
validation_axes[1, 0].set_title("Mean probability at target centers — higher is better")
validation_axes[1, 0].set_ylabel("Predicted probability")
validation_axes[1, 0].tick_params(axis="x", rotation=25)

for model_name, short_name in zip(model_names, short_model_names):
    sorted_distances = np.sort(comparison_results[model_name]["peak_distances"])
    cumulative_fraction = np.arange(1, len(sorted_distances) + 1) / len(sorted_distances)
    validation_axes[1, 1].plot(sorted_distances, cumulative_fraction, label=short_name)
validation_axes[1, 1].axvline(1.0, color="black", linestyle="--", linewidth=1, label="1 pixel")
validation_axes[1, 1].set_title("Cumulative peak-distance distribution")
validation_axes[1, 1].set_xlabel("Distance in heatmap pixels")
validation_axes[1, 1].set_ylabel("Fraction of scenes at or below distance")
validation_axes[1, 1].set_ylim(0.0, 1.0)
validation_axes[1, 1].legend(fontsize=8)

fig.suptitle(f"U-Net checkpoint comparison on {len(evaluated_seeds)} shared random unseen scenes", fontsize=16)
plt.show()
